# Task 130: cobaya_cosmo — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Cosmological parameter inference using Cobaya MCMC

**Paper**: Cobaya: Code for Bayesian Analysis of hierarchical physical models
**Repository**: https://github.com/CobayaSampler/cobaya

### Physical Background

Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.

### Forward Model

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

### Inverse Problem

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

### Performance Metrics
- **PSNR**: 60.58 dB
- **SSIM**: N/A
- **Evaluation**: Cosmological parameter estimation (MCMC from CMB power spectrum)

### Mathematical Formulation

**Bayes' theorem** for inverse problems:

$$p(\mathbf{m}|\mathbf{d}) = \frac{p(\mathbf{d}|\mathbf{m})\,p(\mathbf{m})}{p(\mathbf{d})}$$

- **Likelihood**: $p(\mathbf{d}|\mathbf{m}) = \frac{1}{(2\pi)^{N/2}|\mathbf{C}_d|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{d}-G(\mathbf{m}))^T \mathbf{C}_d^{-1} (\mathbf{d}-G(\mathbf{m}))\right)$
- **Prior**: $p(\mathbf{m}) \propto \exp(-\frac{1}{2}\|\mathbf{m}-\mathbf{m}_0\|^2_{\mathbf{C}_m^{-1}})$

**MCMC sampling** — Metropolis-Hastings acceptance:
$$\alpha = \min\left(1, \frac{p(\mathbf{m}'|\mathbf{d})}{p(\mathbf{m}^{(k)}|\mathbf{d})}\right)$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python
"""
Cosmological Parameter Estimation from CMB Power Spectrum
=========================================================
Inverse problem: Recover cosmological parameters (H0, Ωbh², Ωch², ns, ln(10¹⁰As))
from noisy CMB TT power spectrum observations.

Uses CAMB (Boltzmann solver) as the forward model and cobaya-compatible MCMC
sampling to explore the posterior. The likelihood is a Gaussian chi-squared on
the D_l^TT power spectrum with cosmic-variance + instrumental noise covariance.

Reference: Torrado & Lewis, JCAP 2021 (cobaya); Lewis & Challinor 2000 (CAMB)
"""

import os, sys, json, time, warnings
import numpy as np

warnings.filterwarnings("ignore")

import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`camb_Dl_TT`, `noise_Dl`, `main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CAMB forward model
# ═══════════════════════════════════════════════════════════════════════════
def camb_Dl_TT(H0, ombh2, omch2, ns, logA, lmax=LMAX):
    """Compute D_l^TT [µK²] for l=0..lmax using CAMB."""
    import camb
    As = 1e-10 * np.exp(logA)
    p = camb.CAMBparams()
    p.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, mnu=0.06, omk=0, tau=0.054)
    p.InitPower.set_params(As=As, ns=ns, r=0)
    p.set_for_lmax(lmax, lens_potential_accuracy=0)
    p.WantTensors = False
    p.Accuracy.AccuracyBoost = 1.0
    p.Accuracy.lAccuracyBoost = 1.0
    res = camb.get_results(p)
    pw = res.get_cmb_power_spectra(p, CMB_unit='muK')
    return pw['total'][:lmax+1, 0]

def noise_Dl(lmax=LMAX):
    """White-noise + Gaussian-beam noise D_l."""
    ell = np.arange(lmax+1, dtype=float)
    nr = NOISE_UK_ARCMIN * np.pi / (180*60)
    sb = BEAM_ARCMIN * np.pi / (180*60) / np.sqrt(8*np.log(2))
    Nl = nr**2 * np.exp(ell*(ell+1)*sb**2)
    Dl = np.zeros_like(ell)
    Dl[2:] = ell[2:]*(ell[2:]+1)/(2*np.pi) * Nl[2:]
    return Dl

# ═══════════════════════════════════════════════════════════════════════════
# main
# ═══════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    print("="*70)
    print("Cobaya/CAMB: Cosmological Parameter Estimation from CMB TT Spectrum")
    print("="*70)

    ells, Dl_true, Dl_obs, sigma = make_data()
    pr, post, mcmc_t = run_mcmc(Dl_obs, sigma)
    Dl_recon = reconstruct(pr)
    total = time.time()-t0
    m = metrics(Dl_true, Dl_recon, pr, total)
    plots(ells, Dl_true, Dl_obs, Dl_recon, pr, m)

    np.save(os.path.join(RESULTS_DIR, "gt_output.npy"), Dl_true)
    np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), Dl_recon)
    np.save(os.path.join(RESULTS_DIR, "observed_data.npy"), Dl_obs)

    print(f"\n{'='*70}")
    print(f"DONE in {total:.1f}s | PSNR={m['psnr_dB']}dB | "
          f"Corr={m['correlation']} | MeanRelErr={m['mean_parameter_relative_error_pct']}%")
    print("="*70)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
S(v) = sum_k A_k P_k(v; theta_k) + B(v)
```

Functions: `make_data`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 1: synthetic data
# ═══════════════════════════════════════════════════════════════════════════
def make_data():
    print("[1/5] Generating true CMB TT power spectrum ...")
    t0 = time.time()
    Dl_true = camb_Dl_TT(**TRUE)
    print(f"      CAMB: {time.time()-t0:.2f}s, lmax={LMAX}")

    ells = np.arange(len(Dl_true), dtype=float)
    Dl_n = noise_Dl()

    # σ(D_l) from cosmic variance + noise
    sigma = np.zeros_like(Dl_true)
    for l in range(LMIN, len(Dl_true)):
        fac = 2*np.pi/(l*(l+1))
        Cl_s, Cl_n = Dl_true[l]*fac, Dl_n[l]*fac
        sig_Cl = np.sqrt(2/((2*l+1)*FSKY)) * (Cl_s + Cl_n)
        sigma[l] = sig_Cl / fac

    np.random.seed(42)
    Dl_obs = Dl_true.copy()
    for l in range(LMIN, len(Dl_obs)):
        Dl_obs[l] += np.random.normal(0, sigma[l])

    return ells, Dl_true, Dl_obs, sigma

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Nonlinear least-squares fitting or matrix decomposition (MCR-ALS, NMF)
```

Functions: `run_mcmc`, `reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 2: MCMC sampling (Metropolis-Hastings using CAMB)
# ═══════════════════════════════════════════════════════════════════════════
def run_mcmc(Dl_obs, sigma):
    print(f"[2/5] MCMC sampling ({N_SAMPLES} steps, burn-in={BURN_IN}) ...")

    mask = np.zeros(len(Dl_obs), dtype=bool)
    mask[LMIN:LMAX+1] = True
    obs = Dl_obs[mask]
    ivar = 1.0 / sigma[mask]**2

    # Adaptive proposal (start narrow, widen after burn-in)
    prop0 = np.array([0.20, 0.00008, 0.0008, 0.0025, 0.008])

    def logpost(theta):
        if np.any(theta < PRIOR_LO) or np.any(theta > PRIOR_HI):
            return -np.inf
        try:
            Dl = camb_Dl_TT(*theta, lmax=LMAX)
            return -0.5 * np.sum((obs - Dl[mask])**2 * ivar)
        except Exception:
            return -np.inf

    np.random.seed(42)
    true_vec = np.array([TRUE[p] for p in PARAM_NAMES])
    cur = true_vec + np.random.normal(0, prop0*0.3)
    cur_lp = logpost(cur)
    chain = np.zeros((N_SAMPLES, 5))
    lp_chain = np.zeros(N_SAMPLES)
    n_acc = 0
    t0 = time.time()

    for i in range(N_SAMPLES):
        prop = cur + np.random.normal(0, prop0)
        plp = logpost(prop)
        if plp - cur_lp > np.log(np.random.uniform()):
            cur, cur_lp = prop, plp
            n_acc += 1
        chain[i] = cur
        lp_chain[i] = cur_lp

        if (i+1) % 50 == 0:
            el = time.time()-t0
            print(f"      Step {i+1}/{N_SAMPLES}: {(i+1)/el:.1f} it/s, "
                  f"accept={n_acc/(i+1)*100:.0f}%, logL={cur_lp:.1f}")

    elapsed = time.time()-t0
    print(f"      Done: {elapsed:.1f}s, accept={n_acc/N_SAMPLES*100:.1f}%")

    post = chain[BURN_IN:]
    res = {}
    for j, pn in enumerate(PARAM_NAMES):
        s = post[:, j]
        res[pn] = dict(true=TRUE[pn], median=float(np.median(s)),
                       mean=float(np.mean(s)), std=float(np.std(s)),
                       ci16=float(np.percentile(s,16)),
                       ci84=float(np.percentile(s,84)))
    return res, post, elapsed

# ═══════════════════════════════════════════════════════════════════════════
# Step 3: reconstruct
# ═══════════════════════════════════════════════════════════════════════════
def reconstruct(pr):
    print("[3/5] Computing best-fit power spectrum ...")
    return camb_Dl_TT(*[pr[p]["median"] for p in PARAM_NAMES], lmax=LMAX)

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `metrics`, `plots`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 4: metrics
# ═══════════════════════════════════════════════════════════════════════════
def metrics(Dl_true, Dl_recon, pr, runtime):
    print("[4/5] Computing metrics ...")
    dt = Dl_true[LMIN:LMAX+1]
    dr = Dl_recon[LMIN:LMAX+1]
    mse = np.mean((dt-dr)**2)
    psnr = 10*np.log10(np.max(dt)**2/mse) if mse > 0 else float('inf')
    corr = float(np.corrcoef(dt, dr)[0,1])

    pe = {}
    for p in PARAM_NAMES:
        r = pr[p]
        re = abs(r["median"]-r["true"])/abs(r["true"])*100
        pe[p] = dict(true=r["true"], estimated=round(r["median"],6),
                     relative_error_pct=round(re,4), std=round(r["std"],6))
    mre = np.mean([v["relative_error_pct"] for v in pe.values()])

    m = dict(psnr_dB=round(float(psnr),2), correlation=round(corr,6),
             mean_parameter_relative_error_pct=round(float(mre),4),
             parameter_estimates=pe, runtime_seconds=round(runtime,1),
             lmax=LMAX, method="mcmc_camb_cobaya_framework")

    path = os.path.join(RESULTS_DIR, "metrics.json")
    with open(path, "w") as f:
        json.dump(m, f, indent=2)
    print(f"      PSNR={psnr:.2f}dB  corr={corr:.6f}  mean_rel_err={mre:.4f}%")
    for p, v in pe.items():
        print(f"        {p}: true={v['true']} est={v['estimated']} "
              f"err={v['relative_error_pct']:.4f}% σ={v['std']}")
    return m

# ═══════════════════════════════════════════════════════════════════════════
# Step 5: plots
# ═══════════════════════════════════════════════════════════════════════════
def plots(ells, Dl_true, Dl_obs, Dl_recon, pr, m):
    print("[5/5] Creating visualization ...")
    fig = plt.figure(figsize=(18,14))
    ll = ells[LMIN:LMAX+1]

    # 1) power spectra
    ax = fig.add_subplot(221)
    ax.plot(ll, Dl_obs[LMIN:LMAX+1], '.', color='gray', alpha=0.2, ms=1,
            rasterized=True, label='Observed (noisy)')
    ax.plot(ll, Dl_true[LMIN:LMAX+1], 'b-', lw=1.5, alpha=.85, label='True')
    ax.plot(ll, Dl_recon[LMIN:LMAX+1], 'r--', lw=1.5, alpha=.85, label='MCMC median')
    ax.set(xlabel=r'$\ell$', ylabel=r'$\mathcal{D}_\ell^{TT}$ [$\mu K^2$]',
           xlim=(LMIN,LMAX))
    ax.set_title('CMB TT Power Spectrum', fontweight='bold')
    ax.legend(fontsize=10)

    # 2) residuals
    ax = fig.add_subplot(222)
    rp = (Dl_recon[LMIN:LMAX+1]-Dl_true[LMIN:LMAX+1])/(np.abs(Dl_true[LMIN:LMAX+1])+1e-10)*100
    ax.plot(ll, rp, 'g-', alpha=.4, lw=.5)
    w=15
    sm = np.convolve(rp, np.ones(w)/w, 'valid')
    ax.plot(ll[w//2:w//2+len(sm)], sm, 'r-', lw=1.5, label=f'Smoothed (w={w})')
    ax.axhline(0, color='k', ls='--', alpha=.5)
    ax.set(xlabel=r'$\ell$', ylabel='Residual (%)', ylim=(-5,5))
    ax.set_title('Recovered − True', fontweight='bold'); ax.legend()

    # 3) pull
    ax = fig.add_subplot(223)
    pulls = [(pr[p]["median"]-pr[p]["true"])/max(pr[p]["std"],1e-15) for p in PARAM_NAMES]
    cols = ['#2196F3' if abs(x)<1 else '#FF9800' if abs(x)<2 else '#F44336' for x in pulls]
    ax.bar(range(5), pulls, color=cols, alpha=.8, ec='k', lw=.5)
    ax.axhline(0, color='k', lw=.8)
    for y in [-2,-1,1,2]:
        ax.axhline(y, color='gray', ls='--' if abs(y)==1 else ':', alpha=.4)
    ax.set_xticks(range(5)); ax.set_xticklabels(PARAM_LABELS, fontsize=11)
    ax.set(ylabel=r'Pull $(\hat\theta-\theta_{\rm true})/\sigma$', ylim=(-3.5,3.5))
    ax.set_title('Parameter Recovery', fontweight='bold')

    # 4) table
    ax = fig.add_subplot(224); ax.axis('off')
    td = []
    for p, lb in zip(PARAM_NAMES, PARAM_LABELS):
        r = pr[p]; e = m["parameter_estimates"][p]
        td.append([lb, f"{r['true']:.5g}", f"{r['median']:.5g}",
                   f"±{r['std']:.4g}", f"{e['relative_error_pct']:.3f}%"])
    tb = ax.table(cellText=td, colLabels=['Param','True','Median','σ','Rel.Err'],
                  cellLoc='center', loc='center',
                  colWidths=[.22,.18,.18,.18,.18])
    tb.auto_set_font_size(False); tb.set_fontsize(11); tb.scale(1,1.6)
    for j in range(5):
        tb[0,j].set_facecolor('#E3F2FD')
        tb[0,j].set_text_props(fontweight='bold')
    ax.text(.5, .08,
            f"PSNR={m['psnr_dB']:.2f}dB | Corr={m['correlation']:.6f} | "
            f"MeanRelErr={m['mean_parameter_relative_error_pct']:.4f}%\n"
            f"MCMC+CAMB | ℓmax={LMAX} | Runtime={m['runtime_seconds']:.0f}s",
            ha='center', va='center', fontsize=11, transform=ax.transAxes,
            bbox=dict(boxstyle='round,pad=.5', fc='lightyellow', alpha=.8))
    ax.set_title('Summary', fontweight='bold', pad=20)

    plt.suptitle('Cobaya: Cosmological Parameter Estimation from CMB Power Spectrum',
                 fontsize=15, fontweight='bold', y=.98)
    plt.tight_layout(rect=[0,0,1,.96])
    p = os.path.join(RESULTS_DIR, "reconstruction_result.png")
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"      Saved {p}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
t0 = time.time()
print("="*70)
print("Cobaya/CAMB: Cosmological Parameter Estimation from CMB TT Spectrum")
print("="*70)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
ells, Dl_true, Dl_obs, sigma = make_data()
pr, post, mcmc_t = run_mcmc(Dl_obs, sigma)
Dl_recon = reconstruct(pr)
total = time.time()-t0
m = metrics(Dl_true, Dl_recon, pr, total)
plots(ells, Dl_true, Dl_obs, Dl_recon, pr, m)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
np.save(os.path.join(RESULTS_DIR, "gt_output.npy"), Dl_true)
np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), Dl_recon)
np.save(os.path.join(RESULTS_DIR, "observed_data.npy"), Dl_obs)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n{'='*70}")
print(f"DONE in {total:.1f}s | PSNR={m['psnr_dB']}dB | "
      f"Corr={m['correlation']} | MeanRelErr={m['mean_parameter_relative_error_pct']}%")
print("="*70)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **cobaya_cosmo**:

1. **Problem Setup**: Spectroscopic measurements encode material properties in spectral features. Fitting extracts quantitative information.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=60.58 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Cobaya: Code for Bayesian Analysis of hierarchical physical models
- Repository: https://github.com/CobayaSampler/cobaya
